In [1]:
from Simulation.mpc import *
from Simulation.system_functions import PolymerCSTR
from utils.helpers import *

## Initialize the system

In [2]:
# First initiate the system
# Parameters
Ad = 2.142e17           # h^-1
Ed = 14897              # K
Ap = 3.816e10           # L/(molh)
Ep = 3557               # K
At = 4.50e12            # L/(molh)
Et = 843                # K
fi = 0.6                # Coefficient
m_delta_H_r = -6.99e4   # j/mol
hA = 1.05e6             # j/(Kh)
rhocp = 1506            # j/(Kh)
rhoccpc = 4043          # j/(Kh)
Mm = 104.14             # g/mol
system_params = np.array([Ad, Ed, Ap, Ep, At, Et, fi, m_delta_H_r, hA, rhocp, rhoccpc, Mm])

In [3]:
# Design Parameters
CIf = 0.5888    # mol/L
CMf = 8.6981    # mol/L
Qi = 108.       # L/h
Qs = 459.       # L/h
Tf = 330.       # K
Tcf = 295.      # K
V = 3000.       # L
Vc = 3312.4     # L

system_design_params = np.array([CIf, CMf, Qi, Qs, Tf, Tcf, V, Vc])

In [4]:
# Steady State Inputs
Qm_ss = 378.    # L/h
Qc_ss = 471.6   # L/h

system_steady_state_inputs = np.array([Qc_ss, Qm_ss])

In [5]:
# Sampling time of the system
delta_t = 0.5 # 30 mins

In [6]:
# Initiate the CSTR for steady state values
cstr = PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t)
steady_states={"ss_inputs":cstr.ss_inputs,
               "y_ss":cstr.y_ss}

## Loading the system matrices, min max scaling, and min max of the states

In [7]:
dir_path = os.path.join(os.getcwd(), "Data")

In [8]:
# Defining the range of setpoints for data generation
setpoint_y = np.array([[3.2, 321],
                       [4.5, 325]])
u_min = np.array([71.6, 78])
u_max = np.array([870, 670])

system_data = load_and_prepare_system_data(steady_states=steady_states, setpoint_y=setpoint_y, u_min=u_min, u_max=u_max)

In [9]:
A_aug = system_data["A_aug"]
B_aug = system_data["B_aug"]
C_aug = system_data["C_aug"]

In [10]:
data_min = system_data["data_min"]
data_max = system_data["data_max"]

In [11]:
min_max_states = {'max_s': np.array([256.79686253, 256.01560603,  48.99447186, 144.79949103,
          2.82199733,   3.14014989,   2.78866348,   3.71691422,
          6.2029936 ]),
                  'min_s': np.array([ -272.28060121, -1112.33972595,   -76.63993491,  -608.60327886,
           -3.94399122,    -3.93115257,    -2.9532091 ,    -4.06547624,
          -28.25906582])}

In [12]:
y_sp_scaled_deviation = system_data["y_sp_scaled_deviation"]

In [13]:
b_min = system_data["b_min"]
b_max = system_data["b_max"]

In [14]:
min_max_dict = system_data["min_max_dict"]
min_max_dict["x_max"] = np.array([256.79686253, 256.01560603,  48.99447186, 144.79949103,
          2.82199733,   3.14014989,   2.78866348,   3.71691422,
          6.2029936 ])
min_max_dict["x_min"] = np.array([ -272.28060121, -1112.33972595,   -76.63993491,  -608.60327886,
           -3.94399122,    -3.93115257,    -2.9532091 ,    -4.06547624,
          -28.25906582])

In [38]:
# Setpoints in deviation form
inputs_number = int(B_aug.shape[1])
y_sp_scenario = np.array([[4.5, 324],
                          [3.4, 321]])

y_sp_scenario = (apply_min_max(y_sp_scenario, data_min[inputs_number:], data_max[inputs_number:])
                 - apply_min_max(steady_states["y_ss"], data_min[inputs_number:], data_max[inputs_number:]))
n_tests = 200
set_points_len = 400
TEST_CYCLE = [False, False, False, False, False]
warm_start = 5
ACTOR_FREEZE = 20 * set_points_len
warm_start_plot = warm_start * 2 * set_points_len + ACTOR_FREEZE

In [39]:
# Observer Gain
poles = np.array(np.array([0.44619852, 0.33547649, 0.36380595, 0.70467118, 0.3562966,
                           0.42900673, 0.4228262 , 0.96916776, 0.91230187]))
L = compute_observer_gain(A_aug, C_aug, poles)

## Setting The hyperparameters for the TD3 Agent


In [115]:
from TD3Agent.agent import TD3Agent
import torch

In [184]:
set_points_number = int(C_aug.shape[0])
inputs_number = int(B_aug.shape[1])
STATE_DIM = int(A_aug.shape[0]) + 3 * set_points_number + inputs_number
ACTION_DIM = inputs_number
n_outputs = C_aug.shape[0]
a_coef, b_coef, c_coef = 1, 1, 1
ACTOR_LAYER_SIZES = [512, 512, 512]
CRITIC_LAYER_SIZES = [512, 512, 512]
RES_LOW = np.array([-0.5, -0.5], dtype=float)
RES_HIGH = np.array([0.5, 0.5], dtype=float)
BUFFER_CAPACITY = 80000
ACTOR_LR = 1e-4
CRITIC_LR = 1e-4
SMOOTHING_STD = 0.1
NOISE_CLIP = 0.2
GAMMA = 0.99
TAU = 0.005
MAX_ACTION = 1.0
POLICY_DELAY = 2
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 128
STD_START = 0.2
STD_END = 0.02
STD_DECAY_RATE = 0.99995
STD_DECAY_MODE = "exp"

In [185]:
td3_agent = TD3Agent(
    state_dim=STATE_DIM,
    action_dim=ACTION_DIM,
    actor_hidden=ACTOR_LAYER_SIZES,
    critic_hidden=CRITIC_LAYER_SIZES,
    gamma=GAMMA,
    actor_lr=ACTOR_LR,
    critic_lr=CRITIC_LR,
    batch_size=BATCH_SIZE,
    policy_delay=POLICY_DELAY,
    target_policy_smoothing_noise_std=SMOOTHING_STD,
    noise_clip=NOISE_CLIP,
    max_action=MAX_ACTION,
    tau=TAU,
    std_start=STD_START,
    std_end=STD_END,
    std_decay_rate=STD_DECAY_RATE,
    std_decay_mode=STD_DECAY_MODE,
    buffer_size=BUFFER_CAPACITY,
    device=DEVICE,
    actor_freeze=ACTOR_FREEZE,
    )

## MPC initialization and setpoints

In [186]:
# MPC parameters
n_inputs = 2
predict_h = 9
cont_h = 3
u_ss = apply_min_max(steady_states['ss_inputs'], data_min[:n_inputs], data_max[:n_inputs])
b_min = apply_min_max(np.array([71.6, 78]), data_min[:n_inputs], data_max[:n_inputs])
b_max= apply_min_max(np.array([870, 670]), data_min[:n_inputs], data_max[:n_inputs])
b1 = (b_min[0]-u_ss[0], b_max[0]-u_ss[0])
b2 = (b_min[1]-u_ss[1], b_max[1]-u_ss[1])
bnds = (b1, b2)*cont_h
cons = []
IC_opt = np.zeros(n_inputs*cont_h)
Q1_penalty = 5.
Q2_penalty = 1.
R1_penalty = 1
R2_penalty = 1

In [187]:
MPC_obj = MpcSolverGeneral(A_aug, B_aug, C_aug,
    Q_out=np.array([Q1_penalty, Q2_penalty]),
    R_in=np.array([R1_penalty, R2_penalty]),
    NP=predict_h,
    NC=cont_h)

In [188]:
def make_reward_fn_relative_QR(
    data_min, data_max, n_inputs,
    k_rel, band_floor_phys,
    Q_diag, R_diag,
    tau_frac=0.7,
    gamma_out=0.5, gamma_in=0.5,
    beta=5.0, gate="geom", lam_in=1.0,
    bonus_kind="exp", bonus_k=12.0, bonus_p=0.6, bonus_c=20.0,
):
    """
    Reward with relative tracking bands.

    data_min, data_max : arrays for [u_min..., y_min...], [u_max..., y_max...]
    n_inputs           : number of inputs (so outputs start at index n_inputs)
    k_rel              : per-output relative tolerance factors (same length as outputs)
    band_floor_phys    : per-output minimum band in physical units
    Q_diag, R_diag     : quadratic weights (same as before)
    """

    data_min = np.asarray(data_min, float)
    data_max = np.asarray(data_max, float)
    dy = np.maximum(data_max[n_inputs:] - data_min[n_inputs:], 1e-12)  # phys range for each y

    k_rel = np.asarray(k_rel, float)
    band_floor_phys = np.asarray(band_floor_phys, float)
    Q_diag = np.asarray(Q_diag, float)
    R_diag = np.asarray(R_diag, float)

    # floor in *scaled* coordinates (used if y_sp_phys is not provided)
    band_floor_scaled = band_floor_phys / np.maximum(dy, 1e-12)

    def _sigmoid(x):
        x = np.clip(x, -60.0, 60.0)
        return 1.0 / (1.0 + np.exp(-x))

    def _phi(z, kind=bonus_kind, k=bonus_k, p=bonus_p, c=bonus_c):
        z = np.clip(z, 0.0, 1.0)
        if kind == "linear":
            return 1.0 - z
        if kind == "quadratic":
            return (1.0 - z) ** 2
        if kind == "exp":
            return (np.exp(-k * z) - np.exp(-k)) / (1.0 - np.exp(-k))
        if kind == "power":
            return 1.0 - np.power(z, p)
        if kind == "log":
            return np.log1p(c * (1.0 - z)) / np.log1p(c)
        raise ValueError("unknown bonus kind")

    def reward_fn(e_scaled, du_scaled, y_sp_phys=None):
        """
        e_scaled : output error in scaled deviation space  (same as before)
        du_scaled: input move in scaled deviation space    (same as before)
        y_sp_phys: current setpoint in *physical* units (array len = n_outputs)
        """

        e_scaled = np.asarray(e_scaled, float)
        du_scaled = np.asarray(du_scaled, float)

        # ----- dynamic band based on setpoint -----
        if y_sp_phys is None:
            # fallback: just use the floor
            band_scaled = band_floor_scaled
        else:
            y_sp_phys_arr = np.asarray(y_sp_phys, float)
            # band_phys_i = max(k_rel_i * |y_sp_i|, band_floor_phys_i)
            band_phys = np.maximum(k_rel * np.abs(y_sp_phys_arr), band_floor_phys)
            band_scaled = band_phys / np.maximum(dy, 1e-12)

        tau_scaled = tau_frac * band_scaled

        # ----- inside/outside gate -----
        abs_e = np.abs(e_scaled)
        s_i = _sigmoid((band_scaled - abs_e) / np.maximum(tau_scaled, 1e-12))

        if gate == "prod":
            w_in = float(np.prod(s_i, dtype=np.float64))
        elif gate == "mean":
            w_in = float(np.mean(s_i))
        elif gate == "geom":
            w_in = float(np.prod(s_i, dtype=np.float64) ** (1.0 / len(s_i)))
        else:
            raise ValueError("gate must be 'prod'|'mean'|'geom'")

        # ----- core quadratic costs -----
        err_quad = np.sum(Q_diag * (e_scaled ** 2))
        err_eff = (1.0 - w_in) * err_quad + w_in * (lam_in * err_quad)
        move = np.sum(R_diag * (du_scaled ** 2))

        # ----- linear penalties around band edge -----
        slope_at_edge = 2.0 * Q_diag * band_scaled

        overflow = np.maximum(abs_e - band_scaled, 0.0)
        lin_out = (1.0 - w_in) * np.sum(gamma_out * slope_at_edge * overflow)

        inside_mag = np.minimum(abs_e, band_scaled)
        lin_in = w_in * np.sum(gamma_in * slope_at_edge * inside_mag)

        # ----- bonus near zero error -----
        qb2 = Q_diag * (band_scaled ** 2)
        z = abs_e / np.maximum(band_scaled, 1e-12)
        phi = _phi(z)
        bonus = w_in * beta * np.sum(qb2 * phi)

        # ----- total reward -----
        return (-(err_eff + move + lin_out + lin_in) + bonus) * 0.01

    params = dict(
        k_rel=k_rel,
        band_floor_phys=band_floor_phys,
        band_floor_scaled=band_floor_scaled,
        Q_diag=Q_diag,
        R_diag=R_diag,
        tau_frac=tau_frac,
        gamma_out=gamma_out,
        gamma_in=gamma_in,
        beta=beta,
        gate=gate,
        lam_in=lam_in,
        bonus_kind=bonus_kind,
        bonus_k=bonus_k,
        bonus_p=bonus_p,
        bonus_c=bonus_c,
    )
    return params, reward_fn

In [189]:
n_inputs = 2

dy = data_max[n_inputs:] - data_min[n_inputs:]
y_sp_nom = 0.5 * (data_min[n_inputs:] + data_max[n_inputs:])

k_rel = np.array([0.0003, 0.0003])
band_floor_phys = np.array([0.006, 0.007])

band_phys = np.maximum(k_rel * np.abs(y_sp_nom), band_floor_phys)

scale_factor = 1.0  # use 2.0 for [-1, 1] scaling, 1.0 for [0, 1]
band_scaled = scale_factor * band_phys / dy

q0 = 1.4
Q_diag = q0 / np.maximum(band_scaled ** 2, 1e-12)

print("dy:", dy)
print("y_sp_nom:", y_sp_nom)
print("band_phys:", band_phys)
print("band_scaled:", band_scaled)
print("Q_diag:", Q_diag)

dy: [0.22165278 0.78153727]
y_sp_nom: [  3.83915067 323.21371982]
band_phys: [0.006      0.09696412]
band_scaled: [0.02706937 0.12406845]
Q_diag: [1910.60931855   90.95055189]


In [190]:
Q_diag = np.array([5000., 1000.])          # rounded from the band-based calculation
R_diag = np.array([90., 90.])          # move cost for du_scaled ~ 0.02

n_inputs = 2

print("Band scaled are:")

params, reward_fn = make_reward_fn_relative_QR(
    data_min, data_max, n_inputs,
    k_rel, band_floor_phys,
    Q_diag, R_diag,
    tau_frac=0.7,
    gamma_out=0.5, gamma_in=0.5,
    beta=7.0, gate="geom", lam_in=1.0,
    bonus_kind="exp", bonus_k=12.0, bonus_p=0.6, bonus_c=20.0,
)
print(params)

Band scaled are:
{'k_rel': array([0.0003, 0.0003]), 'band_floor_phys': array([0.006, 0.007]), 'band_floor_scaled': array([0.02706937, 0.00895671]), 'Q_diag': array([5000., 1000.]), 'R_diag': array([90., 90.]), 'tau_frac': 0.7, 'gamma_out': 0.5, 'gamma_in': 0.5, 'beta': 7.0, 'gate': 'geom', 'lam_in': 1.0, 'bonus_kind': 'exp', 'bonus_k': 12.0, 'bonus_p': 0.6, 'bonus_c': 20.0}


In [191]:
nominal_qs = 459
nominal_qi = 108
nominal_hA = 1.05e6
qi_change = 0.85
qs_change = 1.3
ha_change = 0.85

In [192]:
import numpy as np

def compute_band_scaled(
    y_sp_phys, data_min, data_max, n_inputs,
    k_rel, band_floor_phys
):
    data_min = np.asarray(data_min, float)
    data_max = np.asarray(data_max, float)
    dy = np.maximum(data_max[n_inputs:] - data_min[n_inputs:], 1e-12)

    y_sp_phys = np.asarray(y_sp_phys, float)
    k_rel = np.asarray(k_rel, float)
    band_floor_phys = np.asarray(band_floor_phys, float)

    band_phys = np.maximum(k_rel * np.abs(y_sp_phys), band_floor_phys)
    band_scaled = band_phys / dy
    return band_scaled


def residual_outside_weight(e_track_scaled, band_scaled, tau_frac=0.7, kind="geom"):
    e_track_scaled = np.asarray(e_track_scaled, float)
    band_scaled = np.asarray(band_scaled, float)

    abs_e = np.abs(e_track_scaled)
    tau = np.maximum(tau_frac * band_scaled, 1e-12)

    x = (band_scaled - abs_e) / tau
    x = np.clip(x, -60.0, 60.0)
    s = 1.0 / (1.0 + np.exp(-x))  # per-output "inside-ness" in [0, 1]

    if kind == "prod":
        w_in = float(np.prod(s, dtype=np.float64))
    elif kind == "mean":
        w_in = float(np.mean(s))
    elif kind == "geom":
        w_in = float(np.prod(s, dtype=np.float64) ** (1.0 / len(s)))
    else:
        raise ValueError("kind must be 'prod', 'mean', or 'geom'")

    return 1.0 - w_in  # outside weight


def residual_hard_gate(e_track_scaled, band_scaled, gate_mult=1.0):
    return float(np.all(np.abs(e_track_scaled) <= gate_mult * band_scaled))


In [193]:
import warnings
def compute_observer_gain(A, C, desired_poles):
    """
    Compute an observer gain L for the given MPC system using the desired poles.
    Also performs an observability check.

    Parameters:
    -----------
    A, C : np.ndarray
        System Matrices
    desired_poles : np.ndarray
        A vector of desired observer poles.

    Returns:
    --------
    L : np.ndarray
        The observer gain matrix.
    """
    # Compute the observer gain using pole placement
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        obs_gain_calc = signal.place_poles(A.T, C.T, desired_poles, "KNV0")
    L = np.squeeze(obs_gain_calc.gain_matrix).T

    # Check observability
    observability_matrix = control.obsv(A, C)
    rank = np.linalg.matrix_rank(observability_matrix)
    if rank == A.shape[0]:
        # print("The system is observable.")
        pass
    else:
        # print("The system is not observable.")
        pass
    return L

In [194]:
nominal_qs = 459
nominal_qi = 108
nominal_hA = 1.05e6
nominal_cmf = 8.6981
qi_change = 0.8
qs_change = 1.3
ha_change = 0.8
cmf_chnage = 1.05

In [195]:
def generate_setpoints_training_rl_gradually(y_sp_scenario, n_tests, set_points_len, warm_start, test_cycle,
                                             nominal_qi, nominal_qs, nominal_ha, nominal_cmf,
                                             qi_change, qs_change, ha_change, cmf_change):
    # For each scenario, create a block of size (set_points_len, n_outputs)
    blocks = [np.full((set_points_len, y_sp_scenario.shape[1]), scenario)
              for scenario in y_sp_scenario]

    # Concatenate the blocks to form one cycle
    cycle = np.concatenate(blocks, axis=0)
    # Repeat the cycle 'repetitions' times
    y_sp = np.concatenate([cycle] * n_tests, axis=0)

    # Test train scenario
    test_cycle = test_cycle * int(n_tests / len(test_cycle))
    # Try making everything trainable but te end cycle should be only for testing
    test_cycle[-1] = True

    time_in_sub_episodes = set_points_len * len(y_sp_scenario)

    nFE = int(y_sp.shape[0])
    idxs_setpoints = np.arange(time_in_sub_episodes - 1, nFE, time_in_sub_episodes)
    idxs_tests = np.arange(0, nFE, time_in_sub_episodes)
    sub_episodes_changes = np.arange(1, len(idxs_setpoints) + 1)
    sub_episodes_changes_dict = {}
    test_train_dict = {}
    for i in range(len(idxs_setpoints)):
        sub_episodes_changes_dict[idxs_setpoints[i]] = sub_episodes_changes[i]
    for i in range(len(idxs_tests)):
        test_train_dict[idxs_tests[i]] = test_cycle[i]
    warm_start = list(test_train_dict.keys())[warm_start]

    qi = np.linspace(nominal_qi, nominal_qi * qi_change, nFE)
    qs = np.linspace(nominal_qs, nominal_qs * qs_change, nFE)
    ha = np.linspace(nominal_ha, nominal_ha * ha_change, int(nFE / 2))
    ha = np.hstack((ha, np.tile(nominal_ha * ha_change, int(nFE/ 2))))
    cmf = np.linspace(nominal_cmf, nominal_cmf * cmf_change, nFE)

    done_flags = np.arange(set_points_len - 1, nFE, set_points_len)

    return y_sp, nFE, sub_episodes_changes_dict, time_in_sub_episodes, test_train_dict, warm_start, qi, qs, ha, done_flags, cmf

In [196]:
def run_rl_train_disturbance_gradually(
        system, y_sp_scenario, n_tests, set_points_len,
        steady_states, min_max_dict, agent, MPC_obj,
        poles,
        data_min, data_max, warm_start,
        test_cycle,
        high_res, low_res,
        reward_fn,
        nominal_qi, nominal_qs, nominal_ha, nominal_cmf,
        qi_change, qs_change, ha_change, cmf_change,
        mode="disturb"
):

    # --- setpoints generation ---
    y_sp, nFE, sub_episodes_changes_dict, time_in_sub_episodes, test_train_dict, WARM_START, qi, qs, ha, done_flags, cmf = \
        generate_setpoints_training_rl_gradually(
            y_sp_scenario, n_tests, set_points_len, warm_start, TEST_CYCLE,
            nominal_qi, nominal_qs, nominal_ha, nominal_cmf,
            qi_change, qs_change, ha_change, cmf_change
        )

    # inputs and outputs of the system dimensions
    n_inputs = B_aug.shape[1]
    n_outputs = C_aug.shape[0]
    n_states = A_aug.shape[0]

    # Scaled steady states inputs and outputs
    ss_scaled_inputs = apply_min_max(steady_states["ss_inputs"], data_min[:n_inputs], data_max[:n_inputs])
    y_ss_scaled = apply_min_max(steady_states["y_ss"], data_min[n_inputs:], data_max[n_inputs:])

    # Logs
    y_system = np.zeros((nFE + 1, n_outputs))
    y_system[0, :] = system.current_output
    u_mpc = np.zeros((nFE, n_inputs))
    rewards = np.zeros(nFE)
    avg_rewards = []
    yhat = np.zeros((n_outputs, nFE))
    xhatdhat = np.zeros((n_states, nFE + 1))

    # ----- helper ------
    def map_to_bounds(a, low, high):
        return low + ((a + 1.0) / 2.0) * (high - low)

    def map_from_bounds(x, low, high):
        x = np.asarray(x, float)
        return 2.0 * (x - low) / (high - low) - 1.0

    test = False

    # Tunable knobs (start conservative)
    beta_res = np.array([0.5, 0.5], dtype=np.float32)  # residual authority relative to MPC move
    du0_res = np.array([0.001, 0.001], dtype=np.float32)  # floor so residual can act when MPC move is small
    eta_tol = 0.3  # epsilon_i = eta_tol * band_scaled_i

    # Absolute scaled bounds
    u_min_scaled_abs = apply_min_max(np.array([71.6, 78.0]), data_min[:n_inputs], data_max[:n_inputs])
    u_max_scaled_abs = apply_min_max(np.array([870.0, 670.0]), data_min[:n_inputs], data_max[:n_inputs])


    # we need to recalculate the observer gain
    # # Observer Gain
    L = compute_observer_gain(MPC_obj.A, MPC_obj.C, poles)

    for i in range(nFE):
        # train/test phase
        if i in test_train_dict:
            test = test_train_dict[i]

        # Current scaled input & deviation
        scaled_current_input = apply_min_max(system.current_input, data_min[:n_inputs], data_max[:n_inputs])
        scaled_current_input_dev = scaled_current_input - ss_scaled_inputs

        # measured output at time i in scaled deviation space
        y_prev_scaled = apply_min_max(y_system[i, :], data_min[n_inputs:], data_max[n_inputs:]) - y_ss_scaled

        # predicted output at time i from current observer state
        yhat_pred = np.dot(MPC_obj.C, xhatdhat[:, i])
        # innovation (prediction mismatch) and tracking error
        innov = y_prev_scaled - yhat_pred
        innov = np.clip(innov, -2.0, 2.0).astype(np.float32)
        e_track = y_prev_scaled - y_sp[i, :]
        e_track = np.clip(e_track, -2.0, 2.0).astype(np.float32)

        # ---- RL state (scaled) ----
        current_rl_state = apply_rl_scaled(min_max_dict, xhatdhat[:, i], y_sp[i, :], scaled_current_input_dev).astype(np.float32)
        current_rl_state = np.concatenate([current_rl_state, innov, e_track], axis=0)

        # TD3 raw action in [-1, 1]
        if i > (WARM_START + agent.actor_freeze) and not test:
            a_raw = agent.take_action(current_rl_state, explore=True)
        elif i > (WARM_START + agent.actor_freeze) and test:
            a_raw = agent.act_eval(current_rl_state)
        else:
            a_raw = map_from_bounds(np.array([0.0, 0.0]), low_res, high_res)

        # y_sp in physical band
        y_sp_phys = reverse_min_max(y_sp[i, :] + y_ss_scaled, data_min[n_inputs:], data_max[n_inputs:])

        band_scaled = compute_band_scaled(
            y_sp_phys=y_sp_phys,
            data_min=data_min,
            data_max=data_max,
            n_inputs=n_inputs,
            k_rel=k_rel,
            band_floor_phys=band_floor_phys
        ).astype(np.float32)

        eps_i = (eta_tol * band_scaled).astype(np.float32)
        rho = float(np.clip(np.max(np.abs(e_track) / np.maximum(eps_i, 1e-12)), 0.0, 1.0))

        # Solve MPC for current y_sp and model state x0_model
        sol = spo.minimize(
            lambda x: MPC_obj.mpc_opt_fun(x, y_sp[i, :], scaled_current_input_dev, xhatdhat[:, i]),
            IC_opt, bounds=bnds, constraints=cons
        )

        # if hasattr(sol, "success") and sol.success:
        #     IC_opt = sol.x.copy()  # warm-start next time

        u_base = sol.x[:n_inputs] + ss_scaled_inputs  # baseline absolute input in scaled space
        # Safety net: make sure baseline is feasible (helps avoid empty intersections)
        u_base = np.clip(u_base, u_min_scaled_abs, u_max_scaled_abs)

        # Current absolute input (scaled)
        u_current = scaled_current_input

        # Baseline move
        delta_u_mpc = (u_base - u_current).astype(np.float32)

        # Residual proposal as a move (delta_u), mapped from actor output
        delta_u_res_raw = map_to_bounds(a_raw, low_res, high_res).astype(np.float32)

        # 1) Relative-to-MPC magnitude bound
        mag = (rho * beta_res) * (np.abs(delta_u_mpc) + du0_res)
        low_mag = -mag
        high_mag = mag

        # 2) Headroom bound around baseline input
        low_safe = (u_min_scaled_abs - u_base).astype(np.float32)
        high_safe = (u_max_scaled_abs - u_base).astype(np.float32)

        # Intersection
        low_bound = np.maximum(low_mag, low_safe)
        high_bound = np.minimum(high_mag, high_safe)

        # If something is numerically weird, force residual to zero safely
        bad = low_bound > high_bound
        if np.any(bad):
            low_bound[bad] = 0.0
            high_bound[bad] = 0.0

        # Constrained residual move
        delta_u_res = np.clip(delta_u_res_raw, low_bound, high_bound).astype(np.float32)

        # Apply: u_applied = u_current + delta_u_mpc + delta_u_res = u_base + delta_u_res
        u_applied_scaled_abs = u_base + delta_u_res
        u_applied_scaled_abs = np.clip(u_applied_scaled_abs, u_min_scaled_abs, u_max_scaled_abs).astype(np.float32)

        # Log applied input
        u_mpc[i, :] = u_applied_scaled_abs


        # Executed move for reward
        delta_u = (u_applied_scaled_abs - u_current).astype(np.float32)

        # Executed residual action for replay (what actually happened relative to MPC baseline)
        delta_u_res_exec = (u_applied_scaled_abs - u_base).astype(np.float32)

        # Convert executed residual back to normalized action space for TD3 storage
        a_exec = map_from_bounds(delta_u_res_exec, low_res, high_res).astype(np.float32)
        a_exec = np.clip(a_exec, -1.0, 1.0).astype(np.float32)

        # Plant input in physical units
        u_plant = reverse_min_max(u_applied_scaled_abs, data_min[:n_inputs], data_max[:n_inputs])

        # For observer update
        next_u_dev = u_applied_scaled_abs - ss_scaled_inputs

        # Apply to plant and step
        system.current_input = u_plant
        system.step()
        if mode == "disturb":
            # disturbances
            system.hA = ha[i]
            system.Qs = qs[i]
            system.Qi = qi[i]
            system.CMf = cmf[i]

        # Record output
        y_system[i+1, :] = system.current_output

        # ----- Observer & model roll -----
        y_current_scaled = apply_min_max(y_system[i+1, :], data_min[n_inputs:], data_max[n_inputs:]) - y_ss_scaled

        # Calculate Delta y in deviation form
        delta_y = y_current_scaled - y_sp[i, :]

        # Calculate the next state in deviation form
        yhat[:, i] = yhat_pred
        xhatdhat[:, i + 1] = (
            MPC_obj.A @ xhatdhat[:, i]
            + MPC_obj.B @ next_u_dev
            + L @ (y_prev_scaled - yhat[:, i]).T
        )

        # Next state calculation
        yhat_next_pred = MPC_obj.C @ xhatdhat[:, i + 1]
        innov_next = y_current_scaled - yhat_next_pred
        innov_next = np.clip(innov_next, -2.0, 2.0).astype(np.float32)
        e_track_next = y_current_scaled - y_sp[i, :]
        e_track_next = np.clip(e_track_next, -2.0, 2.0).astype(np.float32)

        # Reward Calculation
        reward = reward_fn(delta_y, delta_u, y_sp_phys)

        # Record rewards
        rewards[i] = reward

        # ----- Next state for TD3 -----
        next_u_dev = u_mpc[i, :] - ss_scaled_inputs
        next_rl_state = apply_rl_scaled(min_max_dict, xhatdhat[:, i + 1], y_sp[i, :], next_u_dev).astype(np.float32)
        next_rl_state = np.concatenate([next_rl_state, innov_next, e_track_next], axis=0)

        # Episode boundary (treat each setpoint block as an episode end)
        if i in done_flags:
            done = 1.0
        else:
            done = 0.0

        # Buffer + train (skip if in test phase)
        if not test:
            agent.push(current_rl_state,
                       a_exec.astype(np.float32),
                       float(reward),
                       next_rl_state,
                       float(done))
            # if i >= warm_start:
            _ = agent.train_step()  # returns loss or None

        # diagnostics at sub-episode boundary
        if i in sub_episodes_changes_dict:
            avg_rewards.append(np.mean(rewards[max(0, i - time_in_sub_episodes + 1): i + 1]))
            print('Sub_Episode:', sub_episodes_changes_dict[i], '| avg. reward:', avg_rewards[-1])
            if hasattr(agent, "_expl_sigma"):
                print('Exploration noise:', agent._expl_sigma)

    # Reverse scaling for RL inputs
    u_rl = reverse_min_max(u_mpc, data_min[:n_inputs], data_max[:n_inputs])

    return (y_system, u_rl, avg_rewards, rewards, xhatdhat, nFE, time_in_sub_episodes,
            y_sp, yhat)

In [ ]:
cstr = PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t)
(y_system, u_rl, avg_rewards, rewards, xhatdhat, nFE, time_in_sub_episodes,
            y_sp, yhat) = run_rl_train_disturbance_gradually(
        cstr, y_sp_scenario, n_tests, set_points_len,
        steady_states, min_max_dict, td3_agent, MPC_obj,
        poles,
        data_min, data_max, warm_start,
        TEST_CYCLE,
        RES_HIGH, RES_LOW,
        reward_fn,
        nominal_qi, nominal_qs, nominal_hA, nominal_cmf,
        qi_change, qs_change, ha_change, cmf_chnage,
        mode="nominal"
)

In [181]:
from utils.plotting import plot_rl_results_td3_multipliers_dqnstyle, compare_mpc_rl_nominal_from_dirs, \
    compare_mpc_rl_disturb_from_dirs

In [182]:
out_td3 = plot_rl_results_td3_multipliers_dqnstyle(
    y_sp=y_sp,
    steady_states=steady_states,
    nFE=nFE,
    delta_t=delta_t,
    time_in_sub_episodes=time_in_sub_episodes,
    y_rl=y_system,
    u_rl=u_rl,
    avg_rewards=avg_rewards,
    data_min=data_min,
    data_max=data_max,
    reward_fn=reward_fn,
    coef_alpha=None,
    coef_delta=None,
    low_coef=RES_LOW,
    high_coef=RES_HIGH,
    start_episode=2,
    prefix_name="td3_multipliers_disturb",
    directory=dir_path,
    save_pdf=False
)

In [183]:
out_dir_cmp = compare_mpc_rl_disturb_from_dirs(
    rl_dir=out_td3,
    mpc_path_or_dir=r"C:\Users\HAMEDI\Desktop\RL_assisted_MPC_polymer\Data\mpc_results_dist_1.pickle",
    reward_fn=reward_fn,
    directory=r"Result",
    prefix_name="disturb_compare_td3_multipliers",
    start_episode=1
)